In [1]:
%pip install ultralytics

from pathlib import Path
import shutil
import yaml
import os
from ultralytics import YOLO

# Note: the 'YOLO' package is not required; the ultralytics package provides the YOLO class.
# If you still see "ModuleNotFoundError: No module named 'ultralytics'" after running this cell,
# restart the kernel (Runtime -> Restart Kernel) and re-run the notebook cells.


  Using cached ultralytics-8.3.229-py3-none-any.whl.metadata (37 kB)
  Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (19 kB)
  Using cached torch-2.9.1-cp313-cp313-win_amd64.whl.metadata (30 kB)
  Using cached torchvision-0.24.1-cp313-cp313-win_amd64.whl.metadata (5.9 kB)
  Using cached ultralytics_thop-2.0.18-py3-none-any.whl.metadata (14 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
Using cached ultralytics-8.3.229-py3-none-any.whl (1.1 MB)
Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl (39.0 MB)
Using cached torch-2.9.1-cp313-cp313-win_amd64.whl (110.9 MB)
Using cached torchvision-0.24.1-cp313-cp313-win_amd64.whl (4.3 MB)
Using cached ultralytics_thop-2.0.18-py3-none-any.whl (28 kB)
Using cached fsspec-2025.10.0-py3-none-any.whl (200 kB)
Using cached networkx-3.5-py3-none-any.whl (2.0 MB)
Using 

In [9]:
PROJECT_DIR = Path.cwd().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd()
OD_DIR = PROJECT_DIR / "data" / "Object Detection Dataset"
# expected:
# OD_DIR/images/{train,val,test}
# OD_DIR/labels/{train,val,test}

for sub in ["images/train","images/val","images/test","labels/train","labels/val","labels/test"]:
    p = OD_DIR / Path(sub)
    print(sub, "exists?" , p.exists(), "count:", sum(1 for _ in p.glob("*")) if p.exists() else 0)

data_yaml = {
    "path": str(OD_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "bird", 1: "drone"}
}

# Create OD_DIR if it doesn't exist
OD_DIR.mkdir(parents=True, exist_ok=True)

with open(OD_DIR/"data.yaml","w") as f:
    yaml.dump(data_yaml, f)
print("Wrote", OD_DIR/"data.yaml")


images/train exists? False count: 0
images/val exists? False count: 0
images/test exists? False count: 0
labels/train exists? False count: 0
labels/val exists? False count: 0
labels/test exists? False count: 0
Wrote f:\Aerial Object Classification & Detection\data\Object Detection Dataset\data.yaml


In [13]:
from pathlib import Path
import yaml, sys

PROJECT_DIR = Path(r"F:/Aerial Object Classification & Detection")
# try to find the object detection dataset folder (common variants)
candidates = [
    PROJECT_DIR / "data" / "object_detection_dataset",
    PROJECT_DIR / "data" / "Object Detection Dataset",
    PROJECT_DIR / "data" / "object_detection_Dataset",
    PROJECT_DIR / "data" / "Object_Detection_Dataset",
]
ds = None
for c in candidates:
    if c.exists():
        ds = c
        break

if ds is None:
    # fall back to any folder in data/ that contains images/ and labels/
    parent = PROJECT_DIR / "data"
    for child in parent.iterdir() if parent.exists() else []:
        if (child / "images").exists() and (child / "labels").exists():
            ds = child
            break

if ds is None:
    print("ERROR: Could not locate object detection dataset folder automatically. Please place dataset under:")
    print("  F:/Aerial Object Classification & Detection/data/<your_object_detection_folder>")
    sys.exit(1)

print("Dataset folder:", ds)

# find train/val/test under images
images_dir = ds / "images"
labels_dir = ds / "labels"
# possible split names
def find_split(split_names):
    for name in split_names:
        p = images_dir / name
        if p.exists():
            return name
    return None

train_name = find_split(["train","train/images","train_images"])
val_name   = find_split(["val","validation","val/images","val_images"])
test_name  = find_split(["test","test/images","test_images"])

print("Detected image subfolders -> train:", train_name, " val:", val_name, " test:", test_name)

# if val missing but test exists, use test as val (temporary) or create val folder
if val_name is None:
    print("Warning: val not found under images. Creating an empty 'val' folder (you should add validation images).")
    (images_dir / "val").mkdir(parents=True, exist_ok=True)
    (labels_dir / "val").mkdir(parents=True, exist_ok=True)
    val_name = "val"

# Write data.yaml with relative paths (ultralytics expects relative to this yaml file or absolute)
data_yaml = {
    "path": str(ds.resolve()),
    "train": f"images/{train_name or 'train'}",
    "val": f"images/{val_name}",
    "test": f"images/{test_name or val_name}",
    "names": {0: "bird", 1: "drone"}
}

yaml_path = ds / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f)

print("Wrote data.yaml to:", yaml_path)
print("Contents:")
print(yaml.safe_dump(data_yaml))


Dataset folder: F:\Aerial Object Classification & Detection\data\object_detection_dataset
Detected image subfolders -> train: None  val: None  test: None
Wrote data.yaml to: F:\Aerial Object Classification & Detection\data\object_detection_dataset\data.yaml
Contents:
names:
  0: bird
  1: drone
path: F:\Aerial Object Classification & Detection\data\object_detection_dataset
test: images/val
train: images/train
val: images/val



In [15]:
from pathlib import Path
PROJECT_DIR = Path(r"F:/Aerial Object Classification & Detection")
pt_files = list(PROJECT_DIR.rglob("*.pt"))
print("Found .pt files:", len(pt_files))
for p in sorted(pt_files):
    print(" -", p)

# Show models/yolov8 project folder
yolo_root = PROJECT_DIR / "models" / "yolov8"
print("\nContents of", yolo_root)
if yolo_root.exists():
    for p in sorted(yolo_root.iterdir()):
        print(" >", p.name, "(dir)" if p.is_dir() else "")
        if p.is_dir():
            for sub in sorted(p.rglob("*")):
                print("   ", sub.relative_to(yolo_root))
else:
    print("yolov8 folder not found at", yolo_root)


Found .pt files: 1
 - F:\Aerial Object Classification & Detection\notebooks\yolov8n.pt

Contents of F:\Aerial Object Classification & Detection\models\yolov8
 > run1 (dir)
    run1\args.yaml
    run1\weights
